# 🚨 AIC — All‑in‑One Jupyter / Colab Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/COolAlien35/AIC/blob/main/AIC_all_in_one.ipynb)

This notebook is designed for **judges and reviewers**. It runs end‑to‑end with sensible defaults.

## What it covers

- ✅ **Remote check (no installs)**: verify the live OpenEnv Space endpoints.
- ✅ **Local run**: clone the repo, install deps, run a tiny benchmark + task graders.
- ✅ **Evidence regen**: regenerate plots and summaries from committed logs.
- ⏳ **Optional GPU path**: run GRPO (takes hours; only if you want to reproduce training).

## Canonical URLs

- **Judge Space page**: `https://huggingface.co/spaces/KINGKK007/aic-training`
- **Judge runtime**: `https://kingkk007-aic-training.hf.space`
- **Results dashboard**: `https://huggingface.co/spaces/KINGKK007/aic-results-dashboard`


## 0) Settings

You can leave these defaults unchanged.

In [ ]:
import os

HF_ENV_HOST = os.environ.get("AIC_HF_ENV_HOST", "https://kingkk007-aic-training.hf.space")
REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/COolAlien35/AIC.git")
REPO_DIR = os.environ.get("AIC_REPO_DIR", "/content/AIC")

print("HF_ENV_HOST:", HF_ENV_HOST)
print("REPO_URL:", REPO_URL)
print("REPO_DIR:", REPO_DIR)

## 1) Remote verification (no installs)

This is the fastest “does it work?” check.

It verifies:
- `GET /health`
- `POST /reset`
- `GET /state/{env_id}`

In [ ]:
import json
import textwrap
from urllib.request import Request, urlopen


def _get(url: str) -> dict:
    with urlopen(url, timeout=15) as r:
        return json.loads(r.read().decode())


def _post(url: str, payload: dict) -> dict:
    data = json.dumps(payload).encode()
    req = Request(url, data=data, headers={"Content-Type": "application/json"}, method="POST")
    with urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())


health = _get(f"{HF_ENV_HOST}/health")
print("/health →", health)
assert health.get("status") == "ok", "health check failed"

reset = _post(
    f"{HF_ENV_HOST}/reset",
    {"episode_id": 0, "base_seed": 42, "fault_mode": "cascading_failure"},
)
env_id = reset.get("env_id")
print("/reset env_id →", env_id)
assert env_id, "reset did not return env_id"

state = _get(f"{HF_ENV_HOST}/state/{env_id}")
print("/state.state.step →", state["state"].get("step"))
print("/state.state.health_score →", state["state"].get("health_score"))

obs = reset.get("observation", {})
print("\nObservation (alert_summary_text snippet):")
print(textwrap.shorten(obs.get("alert_summary_text", ""), width=260, placeholder=" …"))

print("\n✅ Remote verification passed")

## 2) Local clone + install

This runs locally in the notebook runtime (Colab VM).

In [ ]:
%%bash
set -euo pipefail

if [ ! -d "$REPO_DIR" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
fi
cd "$REPO_DIR"

echo "Python:"; python --version
python -m pip install -q --upgrade pip
python -m pip install -q -r requirements.txt

echo "\n✅ install complete"

## 3) Tiny CPU-safe run (benchmark + task graders)

This produces quick, judge-friendly outputs without needing a GPU.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"

python scripts/run_pipeline.py verify
python scripts/run_final_benchmark.py --episodes 1
python scripts/score_tasks.py --episodes 1

echo "\n✅ tiny run complete"

## 4) Regenerate training evidence plots (from committed GRPO log)

This regenerates:
- `results/grpo_reward_curve.png`
- `results/grpo_loss_curve.png`
- `results/grpo_kl_curve.png`
- `results/grpo_training_summary.json`

It reads the committed log: `logs/grpo_progress.jsonl`.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"

python scripts/plot_grpo_progress.py
ls -lah results/grpo_*_curve.png results/grpo_training_summary.json

echo "\n✅ evidence regenerated"

## 5) Optional: OpenAI baseline (rubric)

If you want to run the rubric-mandated baseline locally, set `OPENAI_API_KEY` in the Colab secrets/environment.

> This costs API money and is **not required** to verify the environment. Judges can run this themselves.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"

if [ -z "${OPENAI_API_KEY:-}" ]; then
  echo "OPENAI_API_KEY not set. Skipping OpenAI baseline."
  exit 0
fi

python scripts/openai_baseline.py --episodes 1 --max-calls 50
ls -lah results/openai_baseline_scores.json results/openai_baseline_episodes.csv

echo "\n✅ OpenAI baseline ran"

## 6) Optional: run GRPO training (GPU, hours)

Only run this if you have a GPU runtime and time. This is the long part.

- Colab: **Runtime → Change runtime type → GPU**
- Expected wall-clock on a T4 free tier: ~6 hours for the canonical 80 steps.

> If you only want a quick sanity check, do **not** run this cell.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"

python scripts/run_pipeline.py smoke
# Full GRPO is expensive; uncomment to run the full pipeline.
# python scripts/run_pipeline.py full

echo "\n✅ GPU optional stage complete"